# Computational Tool Choice Impacts CRISPR Spacer-Protospacer Detection

Uri Neri\*<sup>1</sup>  
Antonio Pedro Camargo<sup>1</sup>  
Brian Bushnell<sup>1</sup>  
Rick Beeloo<sup>2</sup>  
Simon Roux<sup>1</sup>

1: DOE Joint Genome Institute, Berkeley, CA, USA  
2: Utrecht University, Padualaan 8, Utrecht, NL 3584 CH  
\* Uri Neri (uneri@lbl.gov)

## 1 Abstract

CRISPR (Clustered Regularly Interspaced Short Palindromic Repeats) systems are a fundamental defense mechanism in prokaryotes, where short sequences called spacers are stored in the host genome to recognize and target exogenous genetic elements. Viromics, the study of viral communities in environmental samples, relies heavily on identifying these spacer-target interactions to understand host-virus relationships. However, the choice of sequence search tool to identify putative spacer targets is often overlooked, leading to an unknown impact of downstream inferences in virus-host analysis. Here, we utilize simulated and real datasets to compare popular sequence alignment and search tools, revealing critical differences in their ability to detect potential matches and handle varying degrees of sequence identity between spacers and potential targets. Finally, we provide general guidelines that may inform future research regarding matching, which is a common practice in studying the complex nature of host-MGE interactions.

## 2 Introduction

CRISPR (clustered regularly interspaced short palindromic repeats) systems play a vital role in prokaryotic defense against mobile genetic elements, including viruses, plasmids, and other autonomous genetic elements<sup>[1](#ref-Mojica_2005),[2](#ref-CRISPR_review)</sup>. These systems are organized as arrays in the bacteria or archaea genome, where short sequences called spacers are interspersed between repeated sequences. The spacer sequences within these arrays guide the targeting of invasive genetic elements, allowing for specific defense against these threats<sup>[3](#ref-CRISPR_classification)</sup>. The corresponding locus on the virus genome where the spacer complements is termed “protospacer”. The analysis of spacer-protospacer pairs is essential in understanding the complex interactions between hosts and MGEs<sup>[4](#ref-Edwards2015_phage_host)</sup>. Beyond their natural role in prokaryotic immunity, CRISPR systems have been adapted into powerful gene-editing frameworks for biotechnology and therapeutic applications<sup>[5](#ref-CRISPR_gene_editing_review)</sup>, though the computational considerations for analyzing natural CRISPR-mediated phage-host interactions differ substantially from predicting off-target effects in gene editing contexts, as discussed below.

The identification of genuine host-MGE interactions through spacer-protospacer matching presents unique challenges due to the dynamic nature of these relationships and the complexity of sequence evolution. While matches between spacers and protospacers are often interpreted as evidence of interaction, various biological and technical factors can complicate this interpretation<sup>[4](#ref-Edwards2015_phage_host),[6](#ref-soto_perez_crispr_2019)</sup>.

Several key scenarios can lead to false positive assignments in spacer-protospacer matching. Low complexity sequences can create spurious matches between simple repeat regions (albeit these can be mitigated through complexity filtering such as tantan<sup>[7](#ref-Frith_2010)</sup> or DUST<sup>[8](#ref-Morgulis_2006)</sup>). Another type of potential false positives are highly conserved sequences shared by unrelated MGEs, potentially resulting from horizontal gene transfer between MGEs. The horizontal transfer of CRISPR arrays themselves on mobile elements further requires careful examination of array genomic context (regions outside the CRISPR loci) and phylogenetic analysis. Self-targeting events, where matches occur against the host genome rather than MGEs, necessitate comparison against host genome databases and analysis of targeting context<sup>[9](#ref-Stern_2010)</sup>. Finally, historical acquisition events may not reflect current interactions, requiring consideration of phylogenetic dating, evolution rates and the effects of the protospacers being under selective pressure to mutate (which may reduce the MGE susceptibility to deterioration by the CRISPR system). This is further complicated by the fact that increasing the allowed distance between sequences directly increases the likelihood of identifying non-related sequences as similar (sharing high nucleic identity) to each other.

False negatives present another challenge in spacer-protospacer matching, particularly when dealing with large databases of potential targets. Many alignment and search tools default to reporting only the best (top) matches or the first matches that pass a given threshold for a given query or HSP. This may result in potentially missing additional legitimate matches. Unfortunately, different tools also handle ambiguous or secondary alignments differently: they may be reported completely, reported up to a number or based on relative alignment quality, or omitted. Similarly, cases where a query sequence has multiple equally scoring matches in different reference sequences are not handled uniformly across tools. This limitation becomes increasingly problematic as databases grow larger and more diverse, a single spacer might match (implying a targeting) multiple related MGEs.

Yet despite these variations, the choice of spacer-to-protospacer search or alignment tool is often not deeply considered. Presently, the common option for this task, popularized by Edwards et al and Biswas et al<sup>[4](#ref-Edwards2015_phage_host),[10](#ref-Biswas2013)</sup>, uses BLASTn<sup>[11](#ref-Altschul1990_blast)</sup> with parameters adjusted for short input sequences. However as for most bioinformatic tools, the exact workflow design and parameter choice can impact the outcome, including in sequence analysis. The importance of proper tool usage and parameter interpretation is highlighted by historical examples in bioinformatics. A striking example is the work of Shah et al,<sup>[12](#ref-Shah2018)</sup>, in which they report how certain misunderstandings of BLAST’s `-max_target_seqs` parameter may lead to incorrect assumptions about result completeness, potentially impacting published analyses. Albeit this was later clarified by Madden et al.,<sup>[13](#ref-Madden2018)</sup> (of the blast development team) as an unfortunate combination of a software bug (that were since patched) affecting rare cases, and misconceptions regarding the process BLAST+ uses for tie-breaking (alignments of equal plausibility), and finally a consideration regarding composition base scoring. Apart from the patched bug, the main outcome of this correspondence led to more explicit details in blast documentation (specifically the appendix “Outline of the BLAST process”). Still, this highlights that misconceptions about the expected exhaustiveness of tools’ result-reporting can also lead to incorrect assumptions about the outcome of an analysis. In practice, most bioinformatic tools use various heuristics and optimizations, typically designed with specific use cases in mind. For example, most short-read mappers assume the reference to be the output of a singular assembly - which would imply the reference does not contain extremely redundant copies of the same nucleic regions, or a limited number of very similar sequences (e.g. strain variants, alternative splice variants), and this assumption impacts the way read mapping is computed and results are reported.

The choice of tool and its parameters can significantly impact the detection of these multiple matches, with some tools prioritizing speed over completeness by limiting the number of reported matches, or by other internal heuristics such as seed sequence selection from high occurring sequences being penalized. This trade-off between sensitivity and computational efficiency is especially important to consider as most available tools were designed for different tasks than spacer-protospacer matching (e.g. expression analysis, homology detection, and variant calling), and under different assumptions (such as reference and query sequence size and database size or the nature of the reference source: from a single isolate or metagenomic sample rather than from aggregation of sequences from different sources).

**Computational Foundations of Sequence Similarity:**

From a computer science perspective, biological sequences are represented as strings of characters drawn from finite alphabets: DNA and RNA sequences use the four-letter nucleobase alphabet (A, U/T, G, C), while protein sequences use the twenty-letter amino acid alphabet. Determining sequence similarity thus becomes a string matching problem, where the goal is to find all occurrences of a query string (or similar variants) within a reference string or database, subject to specified constraints on permitted differences. The fundamental challenge lies in defining (and efficiently computing) a meaningful notion of “similarity” between sequences that may have diverged through evolutionary processes including substitutions, insertions, deletions, and rearrangements.

The classical computational approach to sequence alignment employs dynamic programming algorithms, most notably the Needleman-Wunsch algorithm for global alignment<sup>[14](#ref-Needleman1970_global_alignment)</sup> and the Smith-Waterman algorithm for local alignment<sup>[15](#ref-Smith1981_local_alignment)</sup>. These algorithms guarantee optimal alignments under a given scoring scheme but operate with $O(mn)$ time complexity, where $m$ and $n$ are the lengths of the two sequences being compared. When searching a query of length $m$ against a database of total length $N$, exhaustive application of dynamic programming requires $O(mN)$ operations. For modern metagenomic databases where $N$ can exceed $10^{11}$ bases and query sets may contain millions of spacers, this quadratic scaling becomes computationally prohibitive. For instance, searching 3.8 million spacers (total length \$$132 Mbp) against the IMG/VR4 database ($\$79 Gbp) would require approximately $10^{19}$ operations if using exhaustive pairwise comparisons, translating to centuries of computation time even on modern hardware.

Exhaustive methods that guarantee perfect recall (sensitivity = 1.0) within specified distance thresholds do exist for specific use cases. Tools like Sassy employ bit-parallel algorithms based on Myers’ algorithm<sup>[16](#ref-Myers1999_bitparallel)</sup> to achieve exhaustive approximate string matching with arbitrary edit distance thresholds, while indelfree.sh (in bruteforce mode) provides exhaustive hamming distance matching. These approaches are valuable for validation and ground truth establishment on small datasets, but their computational costs scale poorly.

**Heuristic Algorithms and Their Goal-Driven Design:**

To achieve practical performance on large datasets, virtually all widely-used sequence alignment tools employ heuristic algorithms that sacrifice guaranteed completeness for dramatic improvements in speed. These heuristics are fundamentally goal-driven: they are often designed and optimized for specific biological questions and use cases, with algorithmic choices reflecting assumptions about the expected characteristics of both queries and references. Understanding these design constraints is essential when purposing tools for applications outside their intended scope.

Heuristic sequence search tools typically employ multi-stage filtering architectures. BLAST<sup>[11](#ref-Altschul1990_blast)</sup>, perhaps the most widely used sequence search tool, uses a seed-and-extend strategy: it identifies short exact matches (“seeds” or “words”) between query and database sequences, then extends these seeds using gapped alignment only in promising regions. The seed length, extension threshold, and statistical framework (E-values based on extreme value distribution) are all calibrated for detecting homologs across diverse sequence databases. Modern short-read mappers use similar principles but with different optimizations: Bowtie1 employs FM-index data structures enabling efficient exact substring matching followed by backtracking to allow mismatches<sup>[17](#ref-Langmead2009_bowtie)</sup>; Bowtie2 extends this with a multiseed heuristic and affine gap penalties<sup>[18](#ref-Langmead2012_bowtie2)</sup>; Minimap2 uses minimizer-based sparse seeding combined with chaining algorithms to handle long reads with higher error rates<sup>[19](#ref-Li2018_minimap2)</sup>; StrobeAlign employs randstrobes (hash-based linked k-mers) to improve seed specificity<sup>[20](#ref-Sahlin2022_strobealign)</sup>. Tools designed for large-scale homology searches like MMseqs2 use cascaded k-mer filtering: sequences must share sufficient k-mer matches to pass initial filtering before undergoing more expensive alignment<sup>[21](#ref-Steinegger2017_mmseqs2)</sup>.

Critically, these heuristics introduce reporting biases and completeness limitations that vary depending on database composition and query characteristics. Many tools employ early termination strategies, reporting only the top $k$ matches or the first matches passing a threshold, which can lead to missing equally valid (within alignment thresholds set) alternative alignments. Smart seed selection may penalize high-frequency k-mers to reduce computational burden from repetitive regions, potentially causing reduced sensitivity for highly abundant targets. Some tools may assume references derive from single-source assemblies and optimize for unique best-hit assignment rather than comprehensive multi-mapping detection. These design choices, while appropriate for the tools’ intended applications, currently have unknown impact in the context of spacer-protospacer matching, where queries are short (typically within 25-65 bp), searched across diverse reference sequences often comprised of multiple potential hosts genomes and mobile genetic elements (which may share genes), where a comprehensive detection of all valid matches should be considered.

**Distance Metrics and Their Biological Interpretation:**

Tools differ fundamentally in how they measure sequence similarity, employing different distance metrics that reflect distinct evolutionary models. Hamming distance counts only substitutions and requires sequences of equal length, making it appropriate for scenarios where length-changing mutations are rare or highly deleterious. Edit distance (Levenshtein distance) allows insertions and deletions in addition to substitutions, reflecting a broader evolutionary model. Affine gap distance extends edit distance by assigning different penalties to gap opening versus gap extension, better modeling the biological reality that indels often occur in clusters. Finally, some applications require exact matching with zero tolerance for differences.

Importantly, when comparing tools using different distance metrics with the same numeric threshold (e.g., “≤3 mismatches”), edit/gap-affine-based algorithms will naturally report more matches than hamming-based ones because they solve a more permissive computational problem (). This reflects different definitions of sequence similarity rather than differences in tool quality. Hence, the choice of distance metric should be driven by the biological question and system being studied.

**Distance Metric Choice and Experimental Evidence:**

For CRISPR spacer-protospacer matching in natural systems, the choice between hamming distance and edit distance has both biological and computational implications. This benchmark addresses spacer-protospacer matching in the context of inferring historical phage-host interactions in natural prokaryotic populations, which differs fundamentally from predicting CRISPR off-target effects in gene editing applications. In natural systems, we aim to identify evolutionary relationships since protospacer acquisition, where sequence divergence reflects selective pressure on MGEs to mutate and “escape” host defenses. For gene editing applications, even partial base-pairing (including alignments with indels) can cause unwanted off-target cleavage, necessitating more permissive distance metrics. Similarly, when working with low-accuracy sequencing data (e.g., Oxford Nanopore R9 chemistry<sup>[22](#ref-Jain_2016)</sup>) or analyzing raw reads rather than assembled contigs (made from sufficient sequencing depth), some tolerance for indels may be necessary to account for sequencing errors, as the alignment quality cannot exceed the underlying data quality.

Experimental studies consistently report that phage escape mutations from CRISPR immunity are predominantly single nucleotide substitutions, particularly in the PAM-proximal “seed” region where mismatches have the strongest effect on targeting. Foundational work by Deveau et al.<sup>[23](#ref-Deveau2008)</sup> demonstrated that phages escape CRISPR immunity in *Streptococcus thermophilus* through point mutations in protospacers. Semenova et al.<sup>[24](#ref-Semenova2011)</sup> established that in *E. coli* type I-E CRISPR-Cas system, a seven-nucleotide seed region immediately following the PAM is critical for targeting, with mutations in this seed region abolishing immunity by reducing crRNA-guided Cascade complex binding affinity. Fineran et al.<sup>[25](#ref-Fineran2014)</sup> further showed that phages readily escape through point mutations in the PAM or seed region. More recently, Schelling et al.<sup>[26](#ref-Schelling2023)</sup> demonstrated that phage escape occurs mainly through mutations in PAM and seed regions, with preexisting mismatches at any target location accelerating emergence of mutant phages. Across these experimental systems, escape mutations are consistently reported as single nucleotide polymorphisms rather than indels. Bacterial (the host) mutation rates typically have indels occurring approximately 10×<sup>[27](#ref-Lee_2012_mutation_rate_Ecoli)</sup> less frequently than substitutions (albeit some extreme outliers have been reported such as ~3× less likely (26% of total mutations) in *Acidobacterium capsulatum*<sup>[28](#ref-Kucukyildirim_2021_high_indel_rate)</sup>), and this trend is expected to be similar or more pronounced in phage genomes. Similar to their host, phage transcriptional units often contain several genes with small intragenic spaces<sup>[29](#ref-Hatfull_2011_bacteriophages_genomes)</sup>, often resulting in a particularly coding-dense genome with coding genes occupying most of the genomes (e.g. as demonstrated by Ha et al for multiple diverse phage families,coding sequences occupy on average \>\$92.4% of the entire genome<sup>[30](#ref-Ha_2018)</sup>). The lower indel rates align with the fact that while frameshift-inducing indels are particularly deleterious, substitutions may affect only a single amino acid residue. Frame-preserving indels (multiples of 3 bp) are extremely rare but, when observed, may be particularly strong indicators of selection. We acknowledge that most existing literature focuses on substitutions, potentially stemming from substitutions being easier to detect and characterize than indels, or a potential “assumption of expected” bias where the lack of reports about indel escape mutations may not translate to it being a less frequent phenomena. Indeed, a systematic quantitative comparisons of mutation type frequencies across diverse phage-host systems remain lacking. The only report of a verified indel escape mutation we were able to find is from a study by Paez-Espino et al,<sup>[31](#ref-Paez_Espino_2015)</sup>. In that long-term coevolution experiment with *S. thermophilus* phage 2972, the authors note in the methods section “Finally, postassembly as well as comparative analyses were performed to identify SNPs, indels, and recombination events”, however indels (or gaps) are not mentioned in the main text discussing escape mechanisms, and only a single indel event is listed in the supplemental “Table S5. Phage 2972 targeting” among the escape mutations identified, suggesting even this relatively large experimental setup is not adequate to observe enough varied mutations required for statistical analysis. Of note, the authors report, another type of esacpe mutation - large genomic rearrangments and recombination events. Viral genomes are considered higly mosaic, where genes are commonly exchanged between which are particularly common in phage genomes<sup>[29](#ref-Hatfull_2011_bacteriophages_genomes),[32](#ref-Kupczok_2018_phage_genome_evolution)</sup>. While such rearrangement escape mutation are particuarly interesting, we argue that in the context of sequence search tools, these would not be detectable under either hamming nor edit distance metrics, as the original biologically targeted region is either discarded (replaced by protein of similar function, but not necceraly similar nucleic sequence), or split and repositioned in different loci (in the case of internal rearrangement, such as the original spacer targeted the edge between two subsequent genes that underwent synteny altering rearrangement).

**False Positives in Sequence Similarity Searches:**

A critical consideration in sequence similarity searches is the expected rate of spurious matches arising by chance rather than true biological relationships. Traditionally, false positives in sequence similarity searches are considered as matches that appear similar by standard alignment metrics but arise from convergent evolution, random sequence similarity, or compositional biases rather than common ancestry or functional relationships (note: in our Methods and Results sections we define false positives very differently, as we lack ground truth for evolutionary relationships; see Methods). For random DNA sequences of equal nucleotide composition, the probability of finding an exact match of length $L$ is approximately $(1/4)^L$, suggesting exact matches should be extremely rare. However, this simple model fails to capture biological reality: sequences are not random and exhibit compositional biases (GC content variation), low-complexity regions (simple repeats, homopolymers), and conserved functional elements that can create spurious similarities, or similarities arising from convergent evolution rather than shared ancestry.

Most sequence search tools employ statistical frameworks to estimate false positive rates. BLAST calculates E-values representing the expected number of matches with a given score occurring by chance in a database of specified size, based on extreme value distribution theory<sup>[11](#ref-Altschul1990_blast)</sup>. This framework assumes sequences are i.i.d. (independent and identically distributed) random samples, an assumption violated by real biological sequences. Low-complexity filtering tools like DUST<sup>[8](#ref-Morgulis_2006)</sup> and tantan<sup>[7](#ref-Frith_2010)</sup> attempt to mask repetitive regions that contribute disproportionately to spurious matches. However, determining what constitutes a “true” versus “false” positive is particulary chellenging in the absence of ground truth. For short sequences, this is further complicated: firstly, less positions in the alignment imply less information (e.g. the difference in likelihood as more positions are similar in both subject and query), and secondly, as the space of possible matches is very large even under low distance thresholds, and these are often searched in large genomic databases. Noteably, both viral and spacer databases have been routinely increasing in size in recent years: as CRISPR spacer databases have grown from 366,799 unique spacers in 2017<sup>[33](#ref-Shmakov_2017)</sup> to 3,835,942 in 2023<sup>[34](#ref-camargo_img_vr4_2023)</sup>. This increase is largely attributed to progress in metagenomic sequencing.

**Computational Resource Considerations:**

Another important consideration is computational resource requirements. Memory, storage, and availability of CPU cores are factors differing between tools. Parameter choice may also impact these factors considerably, with certain tools offering tunable parameters to trade-off between sensitivity and computational efficiency. In recent years, spacer database size has been rapidly increasing - from 366,799 unique spacers in 2017<sup>[33](#ref-Shmakov_2017)</sup> to 1,173,006 unique spacers reported in 2021<sup>[35](#ref-Dion_2021)</sup> to 3,835,942 unique spacers in 2023<sup>[34](#ref-camargo_img_vr4_2023)</sup>. Similarly, public virus and MGE databases are growing rapidly, with large contributions from metagenomic samples resulting in routine fold increases in the number of predicted viral contigs<sup>[34](#ref-camargo_img_vr4_2023)</sup>. Most tools require more resources as the size of the database grows, and as this trend continues, certain workflows and tools may become prohibitively expensive to run in a reasonable time frame.

## 3 Methods

### 3.1 Tool Selection

We evaluated several widely-used sequence alignment and search tools, spanning different algorithmic approaches and computational strategies (table 1). The tools were selected based on their availability, historical use in sequence analysis, and diversity of algorithmic approaches. The selection includes both exhaustive methods (Sassy, indelfree.sh in bruteforce mode) that guarantee finding all matches within specified distance thresholds, and heuristic methods that use various optimizations for improved speed.

It is important to note that most of these tools were not specifically designed for CRISPR spacer-protospacer matching, but rather for more general sequence search tasks (MMseqs2, BLASTn-short), alignment/mapping of short reads to reference genomes (Bowtie1, Bowtie2, Minimap2, MUMmer4, StrobeAlign, X-mapper), or versatile pattern matching (Sassy, indelfree.sh). Our focus is specifically on spacer-to-protospacer sequence matching as a bioinformatics task, and we did not evaluate integrated host-prediction tools like SpacePHARER<sup>[36](#ref-Zhang_2021)</sup> or iPHoP<sup>[37](#ref-Roux2023_iphop)</sup>, which perform additional analyses such as phylogenetic evaluation or LCA determination from multiple spacer-protospacer matches.

All tools were configured to maximize sensitivity and avoid artificial limitations on multiple match detection. Some tools required specific parameter adjustments to enable detection of short sequences (e.g., BLASTn-short task, Bowtie1/2 short read modes) or to report all matches rather than only top hits. The exhaustive tools (Sassy, indelfree.sh bruteforce mode) were included specifically to validate the completeness of heuristic tool results on smaller datasets where computational costs remain feasible.

<table>
<thead>
<tr>
<th style="text-align: left;">Aligner</th>
<th style="text-align: center;">Indexing</th>
<th style="text-align: left;">Main Algorithm</th>
<th style="text-align: left;">Heuristic/Exhaustive</th>
<th style="text-align: left;">Reporting/Limiting Threshold Used in Benchmark</th>
<th style="text-align: center;">Year</th>
<th style="text-align: left;">Original Purpose /<br />
intended use</th>
<th style="text-align: left;">Notes</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><a href="https://github.com/BenLangmead/bowtie">Bowtie1</a></td>
<td style="text-align: center;">Yes</td>
<td style="text-align: left;">FM-Index (BWT)</td>
<td style="text-align: left;">Yes (backtracking)</td>
<td style="text-align: left;">Hamming distance</td>
<td style="text-align: center;">2009</td>
<td style="text-align: left;">Short read mapping</td>
<td style="text-align: left;">Optimized for 25-50 bp reads (max 1kbp); ungapped alignment only; backtracking heuristic limits to 3 mismatches</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/BenLangmead/bowtie2">Bowtie2</a></td>
<td style="text-align: center;">Yes</td>
<td style="text-align: left;">FM-Index (BWT)</td>
<td style="text-align: left;">Yes (multiseed + extend)</td>
<td style="text-align: left;">Affine/Edit distance</td>
<td style="text-align: center;">2012</td>
<td style="text-align: left;">Read mapping</td>
<td style="text-align: left;">Uses FM-index for seeding with SIMD-accelerated DP extension; supports gapped, local, and end-to-end alignment</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/lh3/minimap2">Minimap2</a></td>
<td style="text-align: center;">Optional</td>
<td style="text-align: left;">Minimizer + chaining</td>
<td style="text-align: left;">Yes (minimizer seeding)</td>
<td style="text-align: left;">Edit distance</td>
<td style="text-align: center;">2018</td>
<td style="text-align: left;">Long read mapping</td>
<td style="text-align: left;">Lexicographically smallest k-mer per window; collinear chaining with gap penalties; versatile across read types</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/bbushnell/BBTools/">indelfree.sh</a></td>
<td style="text-align: center;">No</td>
<td style="text-align: left;">Multi-kmer matching</td>
<td style="text-align: left;">Bruteforce mode is exhaustive, while ion “Indexed” mode this can be limited by selected kmer length, query length, and number of substitutions</td>
<td style="text-align: left;">Hamming distance</td>
<td style="text-align: center;">Publically introduced to bbtools September 2025</td>
<td style="text-align: left;">Read mapping</td>
<td style="text-align: left;">BBTools suite is Java based, and will use available memory - so the peak memory reported herein (sourced from SLURM logs) does not equate with “minimal required memory”</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/ksahlin/StrobeAlign">StrobeAlign</a></td>
<td style="text-align: center;">Yes</td>
<td style="text-align: left;">Randstrobes</td>
<td style="text-align: left;">Yes (syncmer thinning)</td>
<td style="text-align: left;">Edit distance</td>
<td style="text-align: center;">2022</td>
<td style="text-align: left;">Read mapping</td>
<td style="text-align: left;">Uses hash-based linked strobes (randstrobes) with multi-context seeds (MCS) for hierarchical search</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://blast.ncbi.nlm.nih.gov/doc/blast-help/downloadblastdata.html">BLAST+</a></td>
<td style="text-align: center;">Optional</td>
<td style="text-align: left;">Hit-and-extend</td>
<td style="text-align: left;">Yes (contiguous word)</td>
<td style="text-align: left;">E-value, bit score</td>
<td style="text-align: center;">2009</td>
<td style="text-align: left;">Sequence search</td>
<td style="text-align: left;">BLASTN-short mode uses 11-mer seeds; reports matches based on e-value (expected hits by chance given DB size)</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/soedinglab/MMseqs2">MMseqs2</a></td>
<td style="text-align: center;">Optional</td>
<td style="text-align: left;">K-mer prefiltering</td>
<td style="text-align: left;">Yes (3-stage cascade)</td>
<td style="text-align: left;">E-value, bit score</td>
<td style="text-align: center;">2017</td>
<td style="text-align: left;">Sequence search</td>
<td style="text-align: left;">Double k-mer matching → vectorized ungapped → gapped SW; optimized for many-against-many searches</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/RagnarGrootKoerkamp/sassy">Sassy</a></td>
<td style="text-align: center;">No</td>
<td style="text-align: left;">Uses a bit-parallel algorithm based on Myers’ bitpacking to perform exhaustive approximate string matching (ASM)</td>
<td style="text-align: left;">Exhaustive</td>
<td style="text-align: left;">Edit distance</td>
<td style="text-align: center;">2025</td>
<td style="text-align: left;">Versatile pattern matching, suggested for use in CRISPR (gene-editing) off target detection and raw-read alignments</td>
<td style="text-align: left;">Guarantees perfect recall; explores full edit distance landscape; supports arbitrary distances; high computational cost, requires SIMD instructions (AVX2 and NEON), which most modern CPUs support</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/mathjeff/Mapper">X-mapper</a></td>
<td style="text-align: center;">Yes</td>
<td style="text-align: left;">Gapped x-mer pyramid</td>
<td style="text-align: left;">Yes (dynamic seeds)</td>
<td style="text-align: left;">Edit distance</td>
<td style="text-align: center;">2024</td>
<td style="text-align: left;">Read mapping</td>
<td style="text-align: left;"><strong>Preprint</strong> - Dynamic-length gapped x-mers with “pyramid walking” to optimize seed specificity</td>
</tr>
<tr>
<td style="text-align: left;"><a href="https://github.com/mummer4/mummer">mummer4</a></td>
<td style="text-align: center;">Optional</td>
<td style="text-align: left;">48-bit suffix array</td>
<td style="text-align: left;">Yes (MUM-based)</td>
<td style="text-align: left;">Edit distance</td>
<td style="text-align: center;">2018</td>
<td style="text-align: left;">Genome alignment</td>
<td style="text-align: left;">Identifies Maximal Unique Matches (MUMs) using enhanced suffix array; handles genomes up to 141 Tbp</td>
</tr>
</tbody>
</table>

Table 1: Evaluated Tools and Their Characteristics. **Indexing:** “Yes” indicates a pre-computed index is required (in our benchmark, if an index has to be pre-created the time to construct it is measured along the search/alignment step); “Optional” means a persistent index can be generated for reuse but the tool can also build it on-the-fly for single runs; “No” means no indexing is required or supported. **Heuristic/Exhaustive:** Exhaustive methods guarantee finding all matches within the specified distance threshold; heuristic methods use optimizations (seed-based indexing, chaining, k-mer filtering) to improve speed but may miss some matches. **Reporting/Limiting Threshold:** The primary metric used to report or filter alignments. Hamming distance counts only substitutions; Edit distance allows insertions and deletions; Affine distance uses gap penalties; E-value represents expected matches by chance given database size; Exact match requires perfect identity. Note that edit/affine-based algorithms will naturally report more matches than hamming-based ones when both use the same numeric threshold—this reflects different computational problems being solved rather than tool quality differences. **Tool Configuration:** All tools were configured to maximize sensitivity and avoid artificial limitations on multiple match detection. Exact commands, parameters, and tool versions are provided in Supplementary Table S1.

### 3.2 Data Generation and acquisition

We evaluated tool performance using three complementary approaches with varying levels of ground truth information:

| Dataset Type | Spacers (Count) | Contigs (Count) | Notes |
|:-------------|-----------:|-------------:|:-------------------------------|
| Fully Synthetic | 100k - 500k | 5k - 100k | Generated with varying params (see Supp Table S4) |
| Semi-Synthetic | ~3.8M (Real) | ~420k (Simulated) | Real spacers, synthetic contigs matching IMG/VR4 |
| Real Data (IMG/VR4) | ~3.8M (Real) | ~420k (Real HQ) | Subset of IMG/VR4 v1.1 HQ, filtered iPHoP spacers |

Dataset characteristics and ground truth availability. Synthetic contigs were generated to match IMG/VR4 sequence characteristics (GC content, length distributions). Subsampling of IMG/VR4 HQ contigs enabled inclusion of exhaustive tools on smaller fractions while evaluating all tools on larger fractions and the full HQ dataset.

The sequence statistics for the datasets (spacers and contigs) are detailed in Supplementary Table S4 (**?@tbl-supp-simulation-params**), and the simulation/job-generation setup is summarized in Supplementary Table S4b (**?@tbl-supp-prepare-jobs**). The `fraction_X` datasets represent subsampled fractions of IMG/VR4 HQ contigs, while `ns_X_nc_Y` datasets are synthetic benchmarks. The `ns_3826979_nc_421431_real_baseline` dataset is semi-synthetic, where real spacers are searched against simulated contigs matched to IMG/VR4 characteristics.

#### 3.2.1 Real datasets

To evaluate tool performance in real-world scenarios, we used predicted viral contigs and CRISPR spacers from recent comprehensive databases.

**Viral contigs:** We used the IMG/VR4 v1.1 high-confidence viral contigs<sup>[34](#ref-camargo_img_vr4_2023)</sup>, one of the most comprehensive databases of uncultured phage and viral genomes. These sequences are predicted primarily via geNomad<sup>[38](#ref-camargo_genomad_2024)</sup> scans of metagenomic data, supplemented with sequences from NCBI’s RefSeq and GenBank databases.

To focus on prokaryotic phages and exclude eukaryotic viruses (which typically lack CRISPR systems in their hosts), we applied taxonomic filtering based on ICTV classifications. Specifically, we removed contigs classified into eukaryotic virus families, orders, and classes, including but not limited to major groups such as Adenoviridae, Herpesviridae, Poxviridae, Coronaviridae (families); Herpesvirales, Picornavirales, Bunyavirales (orders); and Megaviricetes, Alsuviricetes, Pokkesviricetes (classes). We additionally filtered out contigs ≤1000 bp to ensure sufficient sequence length for reliable spacer-protospacer matching. After these filtering steps (starting from 5,457,198 high-confidence contigs), the final dataset contains 5,115,894 prokaryotic viral contigs with a total size of ~79 Gbp (range: 1,001 - 2,473,870 bp, median: 7,664 bp, GC%: 44.45%). See Supplementary Table S2 for detailed statistics on the contig dataset and filtering steps.

For benchmarking, we selected a high-quality (HQ) subset of 421,431 contigs (~18.9 Gbp) using stratified sampling to maintain taxonomic class label distributions while focusing on the most reliable viral sequences. This HQ subset served as the base set for subsampling experiments and derived performance analyses. This set is available in the Zenodo deposit of this project<sup>[39](#ref-zenodo_doi)</sup>.

**CRISPR spacers:** We used the curated spacer dataset from iPHoP (June 2025 release)<sup>[37](#ref-Roux2023_iphop)</sup>, which combines CRISPR spacers from both reference genomes and metagenomes. To our knowledge, this dataset represents the largest curated spacers extracted from assembled data and is used in existing host-prediction tools. The raw iPHoP set contain 3,882,812 unique spacers (length range: 25-40 bp, median: 34 bp, GC%: 47.6%) compiled from CRISPR arrays identified primarily via piler-cr<sup>[40](#ref-edgar_piler_cr_2007)</sup> and CRT<sup>[41](#ref-bland_crt_2007)</sup>. We applied minor additional filtering to remove 55833 spacers (~1.4% of all) with low sequence complexity or ambiguity (see below “Complexity filtering”). The entire final set of 3,826,979 spacers was used in all benchmarking analyses pertaining to the “real data”. See Supplementary Table S3 and Supplementary Figure S7 for detailed statistics on the spacer dataset composition and feature distribution.

**Complexity filtering:** In this work, our goal is to benchmark the different tools results when they are executed with the most suitable parameters and settings (often CLI arguments) for the task of spacer-protospacer matching. From that perspective, investigating the effects of different complexity filtering algorithms and implementation is outside the scope of this project. The different tools evaluated handle complexity and ambiguity differently - some have internal, hard-coded restrictions (e.g. blastn does not select seeds from regions with ambiguous (`N`) bases, but allows extending over them from another seed), or provide option to disable complexity filtering (such as `-dust no` CLI option in blastn). Some tools (like sassy) may allow all query sequences to contain Ns, but may allow restricting the target sequence to region with a maximal fraction of N positions. Previous uses of blastn for this task (such as in CRISPRTarget) tend to explicitly disable complexity filtering. Some host-assignment tools (such as iPHoP) employ complexity filtering post-hoc (after collecting the search/alignment tool results). To provide a uniform starting position for all tools, so that the complexity handling is not a confounding factor, we applied a basic complexity filter to remove spacers with low sequence complexity or high ambiguity. We note that these sequences are likely not particularly informative from a biological perspective and may arise from incorrect CRISPR array prediction or extraction, and in some in-house tests for this project, we observed these disproportionately contribute to the computational resource issues associated with a non-informative matches (such as extremely massive output files detailing “potential” alignments to regions of Ns).  
The steps and code to reproduce the filtering are available in the project repository (spacer_inspection.ipynb). Briefly, we first calculated the fraction of each nucleotide (A, T, G, C, N) in each spacer sequence, as well as the GC%, Shannon entropy value, and the number of non-unique 6-mers (i.e., 6-mers that occur more than once in the spacer). We then filtered out spacers with any of the following characteristics: any ambiguous bases (N fraction \> 0), low sequence complexity (Shannon entropy ≤ 1), high homopolymer content (any of A, T, G, C fraction ≥ 0.95), or low k-mer diversity (≥4 non-unique 6-mers). This filtering removed 55,833 spacers (~1.4% of all) and resulted in a final set of 3,826,979 spacers used in all benchmarking analyses pertaining to the “real data”.

**Stratified Subsampling Strategy:** For benchmarking purposes, we selected an initial high-quality (HQ), representative subset of 421,431 contigs from the raw IMG/VR4 dataset as described above, termed “fraction_1”. In this benchmark, we measure the tool results on subsamples of this set for three reasons: first, we can only include the exhaustive tools (Sassy, indelfree.sh bruteforce) on the smaller fraction as they are computioanlly expensive, secondly by comparing the fraction to fraction variation in each tool’s result, we can estimate the tools performance consistency, and thirdly, we can investigate the effect of the dataset (fraction) size on the tools’ resource usage (CPU time, memory). To create these subsamples, we employed a “Representative Sampling” aimed at ensuring the samples reflect the entire sequence population characteristics and diversity. Specifically, each sampled subset had to include representatives from each taxonomic class, at the same proportional quantities the classes had in the 421k contig set. We note that this is a crucial step as the majority of the prokaryotic viruses in IMG/VR4 belong to a handful of classes causing random sampling with small sizes to have few or no representative for the various other viral lineages. Using the stratified sampling method, we created subsets of several different fractions: 0.0005 (279 contigs, 7.04 Mbp), 0.001 (421 contigs, 9.75 Mbp), 0.005 (2,107 contigs, 57.06 Mbp), 0.01 (4,214 contigs, 123.67 Mbp), 0.05 (21,071 contigs, 715.07 Mbp), 0.1 (42,143 contigs, 1.50 Gbp), and 1.0 (full HQ set: 421,431 contigs, 18.87 Gbp). The same set of 3,826,979 spacers was used for all subsamples and the full HQ set. Note that even for the smaller subsamples (0.0005-0.01 fractions), not exhaustive tool completed within reasonable CPU time budgets. Note, in this project, we only include the alignments reported by any tool if that tool finished within the same time limit. A singular exception is blastn for the fraction_1 set, which was allowed to run completion, as it the main point of reference with regards to historical use of (any) tool for this task.

#### 3.2.2 Synthetic dataset generation

To examine each tool’s performance across diverse spacer-to-target matching scenarios, we developed a Rust-based simulation framework accessible through a Python CLI interface with fine-grained control over sequence characteristics. The simulator records the ground truth of all planned spacer occurrences, enabling differentiation between true positives (planned matches) and non-planned matches (validated alignments occurring in unplanned regions but meeting distance thresholds).

**Customizable Sequence Characteristics:** The simulation framework provides several parameters to enable more realistic sequences. Users can specify nucleotide base composition independently for spacers and contigs through either GC content percentages or explicit base frequency parameters (A, T, C, G fractions). Additional parameters control contig and spacer length distributions (uniform or normal), the range of substitution mismatches to introduce when injecting a spacer into simulated contigs, the number or range of times each spacer would be “injected” into a simulated contigs, optional indel mutations (insertion and deletion ranges), and the proportion of spacers to reverse complement. Additionally, a semi-synthetic option is permitted - where either an external (existing) spacer or/and contig set is read from file.

**Sequence Generation Process:** The Rust-based core of the simulator generates DNA sequences using weighted random sampling from the nucleotide alphabet (A, T, C, G) based on the specified base composition. For each position in a sequence, a nucleotide is selected with probability proportional to its configured frequency. Sequence lengths are sampled from the specified distribution type: uniform distributions select lengths with equal probability across the range; normal distributions sample from a Gaussian with mean at the midpoint and standard deviation chosen to span the range; and bell curve distributions use a Beta distribution to create length variation with mode near the center. Before actual sequence modifications begin, the simulator creates an “injection” plan (“injection” refers to replacing a contiguous region of a contig with a spacer sequence, creating what we refer to as “planned spacer occurrences” - this should not be confused with “insertion” indel mutations, which add bases within sequences). This plan predetermines which spacers will be placed into which contigs, how many times each spacer appears, which occurrences will be reverse-complemented, and what mutations each will receive. The plans is then executed across seperate processing threads, so that the load (number of spacers and to generate contigs) is balanced by by total spacer-placement lengths so that a similar value is assigned to each thread. To prevent individual contigs from becoming overly saturated with spacers, the simulator calculates contig utilization - the percentage of total contig base pairs that will be occupied by injected spacers - and reports this value (note - in the simulation run describied below this value remained mostly below 2%). During injection, each spacer replaces an existing contig region of equal length at a randomly selected position, thereby preserving the original contig length while creating the “ground truth” alignments. When injecting spacers into these predetermined positions, the simulator applies mutations in a defined order: first, indels (insertions and deletions) mutations are applied seperatly at random positions (note - this option was not used in the current project); second, substitutions (“mismatches”) replace bases with different nucleotides (preventing identity-preserving substitutions like A→A) at random positions. The number of substitions, the injected spacer coordiantes on the contig, the strand, and the contig and spacer identfiers are recorded as the (planned) ground truth. The final outputs the contigs and spacer seqeuences (as FASTA files), and the ground truth (in tabular format).

**Comparison to real spacers:** Rather than using purely random sequences and uniform distributions, for the sets described here, we configured the synthetic data generation to match certain characteristics of the real-datasets (i.e. the filtered iPHoP spacer set and the HQ IMG/VR4 dataset noted above as “fraction_1”). Specifically, we set the GC content to approximately 49% (spacers), and 46% (contigs), and configured contig lengths to be selected under normal distribution from a realistic range (mostly, 1,501-200,000 bp, see Supplementary Table S4 for complete parameters). To demonstrate the ability of the simulated sequences to mimic the real data sets, we calculated and compared several features (e.g. k-mer repeatability, entropy, base frequencies etc) (see Supplementary Figure S6 and notebooks: spacer_inspection.ipynb). Most analysed features indeed appear similar for the simulated and real sequences, with the exception of that in some complexity measures (k-mer repeatability and LCC), the real spacers have slightly wider range of value, suggesting a minor amount of the real spacers have more extreme values in this regards.

**Simulation Dataset Variants:** We generated multiple synthetic datasets with varying sizes and characteristics to evaluate tool performance under different conditions and to assess the tools consistency (for similar reasons as the real-data subsamples). The datasets follow a naming convention ns\_\[n_spacers\]*nc*\[n_contigs\] indicating spacer and contig counts. Smaller datasets (ns_50000_nc_5000, ns_75000_nc_5000, ns_75000_nc_10000) used 25-40 bp spacers, 10,000-150,000 bp contigs under normal distribution, 1-5 spacer insertions (placements, injections) per each simulated spacer, and 0-5 substitution mismatches, with all tools evaluated at hamming distance ≤5. Medium-sized datasets (ns_100000_nc_10000, ns_100000_nc_20000) used similar parameters with all tools at hamming distance ≤5, while the largest dataset (ns_500000_nc_100000 with 500k spacers and 100k contigs spanning 10,000-550,000 bp) was limited to hamming distance ≤3 and excluded exhaustive tools (Sassy, indelfree bruteforce) due to computational constraints. Additionally, we created a specialized high-insertion-rate dataset (ns_500_nc_5000_HIGH_INSERTION_RATE) with only 500 spacers but 100-2,500 injections per simulated spacer to test tool behavior under extreme multi-mapping scenarios, and a minimal dataset (ns_100_nc_50000) with 100 spacers across 50,000 contigs (2,500-850,000 bp range, 1-3 injections, 25-45 bp spacers) for rapid validation. All synthetic datasets configured spacers and contigs with GC content matching real data (49% and 46% respectively), used normal length distributions, and included 50% reverse-complemented injections to reflect biological reality. See Supplementary Table S4 for complete simulation parameters and resulting dataset statistics. Note, not all tools were able to complete all runs within the same time limit for all subsamples (see “Computational Resource and Runtime Tracking” for details).

#### 3.2.3 Semi-Synthetic dataset

This set uses the existing simulation framework, but instead of generating both spacers and contigs from random sequences, we use the same filtered spacer set as the real-datasets, and generate synthetic contigs matching the sequence characteristics of the HQ IMG/VR4 dataset (fraction_1, see Supplementary Table S2 for dataset details). Note, the spacers are not “injected” into the simulated contigs (using the `--number-spacer-insertions 0 0` option of the `simulate` command). Wr primarily use this set to estimate “non-planned” match rates in a realistic spacer set sequence composition context, and realistic search space context. In this context (0 planned spacer occurrences) we expect all identified matches to reflect chance similarity. We note that this set is considered “large” (by design, similarly to the fraction_1), hence we are only able to use an aggregate of the verified non-exhaustive tools, and only under hamming distance ≤3. This suggests that the actual count and rate may actually be larger. See the “Non-planned Match Rate Estimation” section below for details on the definition of “non-planned” matches.

### 3.3 Coordinate Tolerance and Unique Region Counting

When aggregating results across tools, we implement coordinate tolerance matching to handle slight boundary differences in reported alignments. We observed tools may report alignments with minor variations in start/end coordinates (typically 1-5 bp) due to different handling of terminal mismatches or gaps (see Supplementary Note 1 for detailed example). We use a default 5bp tolerance when merging alignments to count unique spacer-contig regions. This approach reduces double-counting of essentially identical matches, accounts for valid algorithmic differences in gap versus substitution placement at alignment boundaries (or variation in tool-specific “clipping” behavior), which enables fair comparison of tool results coverage (total unique regions detected). All reported alignments from all such regions are verified separably, by extracting the reference contig region and realigning to the spacer sequence (see Alignment Verification section below).

### 3.4 Alignment Verification and Distance Metric Calculation

For comparing alignments across tools, we use hamming distance (counting only substitutions) as our primary distance metric, with biological and computational justification provided below. Our benchmarking CLI tool supports setting thresholds for three distance metrics: (minimal) hamming distance, (minimal) edit distance, and gap-affine (by measuring the edit distance from an alignment with a user provided cost matrix and gap penalties). However, for the analyses presented here, we focus primarily on hamming distance ≤3 for most datasets, with hamming distance ≤5 used in datasets where computational resources permitted. As noted in the introduction, we recommend prioritizing hamming distance for spacer-protospacer matching in the context of phage-host interactions, as this better reflects the predominant mutation types observed in experimental studies of phage escape from CRISPR immunity. To clarify, in the context of gene-editing, a minimal edit distance might serve as a more appropriate metric for off-target prediction. Despite this recommendation, we have attempted to compare hamming and edit distance effects empirically in our analyses (when computationally feasible). We note that while in practice, we observed near complete agreement between the minimal edit and gap-affine distance metrics under the conditions tested, these are not identical measurements - a gap-affine distance metric allows for more flexible gap placement and scoring, aimed at capturing biologically relevant (i.e. sharing a common ancestor) relationships, while the minimal edit distance metric will prioritise the minimal set of edits (substitutions and indels) regardless of their evolutionary likelihood (e.g. the higher rarity of indels compared to substitutions).

**Distance Verification Methodology:** To ensure consistent and accurate distance calculation across tools that use different internal alignment algorithms and scoring schemes, we independently recalculated distances for all reported alignments post-hoc. For the gap-affine distance metric, parasail’s<sup>[42](#ref-Daily2016_parasail)</sup> implementation of the Needleman-Wunsch global alignment algorithm is used, for the minimal edit distance metric, the python version of the edlib library<sup>[43](#ref-Šošić_Šikić_2017_edlib)</sup> is used (`edlib.align`). For the hamming distance metric, we measure the number of non-identical residues in the aligned region. As hamming distance can only be calculated for alignments of the same length, and as some tools report gapped alignments, we first pad the shorter sequence with non-matching characters (e.g. “@”) and then calculate the hamming distance as the count of positions where the two sequences differ. This approach also address potential difference in clipping behavior between tools.

### 3.5 Performance definitions and calculation

We defined the ground truth and classified alignments slightly differently for synthetic and real datasets, as for the larger real-data sets we do not have the results of an exhaustive tool to establish a complete set of alignment.

Speficially, for synthetic datasets, we define three categories: 1. **positive_in_plan:** Alignments matching planned spacer occurrence coordinates (±5bp tolerance, see §sec-coordinate-tolerance). These represent the intended ground truth matches, i.e. sequences were generated following the simulation pre-planned design at known coordinates, strands, and number of mismatches. 2. **positive_not_in_plan:** Tool reported alignments within the allowed distance threshold passing our independent validation, but occurring outside planned regions. These represent chance similarities at the specified distance - not false positives in the technical sense (they are valid alignments), but non-planned matches that may indicate increased background noise under a certain condition (distance threshold, search space size, see the “Non-planned Match Rate Estimation” section below). 3. **invalid_alignment:** Alignments that fail the independent alignment verification, or exceed quality thresholds. These are “true” (in the classical sense) false positives, and we note these tend to result from tool-specific reporting artifacts - not all tools support limiting reported alignments within a specific distance metric and threshold (e.g. strobealign do not have any explicitly option to control what alignments are reported), and the parameters we provide to these tools can only approximate it (such as minimal identity or minimal query coverage). Note that the parameter choice is aiming to include at-least all matches within our analysis scope (hamming ≤3) but often includes a larger range. In the context of this project, we discard these “invalid-alignments” and do not investigate them further.

We extend these definitions to the real datasets by treating all (validated) alignments reported by all tools as “positive_not_in_plan”.  
In this project, we define a tool’s “Recall” as the positive rate, or fraction of positives detected by tool. When this fraction is calculated out of the planned alignments only (not including the non-planned), we specify it by noting “non augmented” (e.g. “non-augmented recall”). This reflects our choice to consider the non-planned validated matches as positives. We argue that from a pure sequence alignment perspective, they are as correct (within the distance threshold) as the planned, and that for the real datasets, where we do not have a complete set of planned matches, this is the only way to calculate recall in practice. Furthermore, we specifically recommend the distance metric and threshold choice (hamming ≤3) as we estimate to by considerably low than the number of real alignments we observe in the real data-set (note, this is not the case for higher distance thresholds or for different distance metrics). Additionally, we note that metrics such as precision ($\frac{\text{TruePositives}}{\text{TruePositives}+\text{FalsePositives}}$) are not applicable in this context - as were we to define the non-planned matches as false positives (for the synthetic sets),we could control this value by adjusting the simulation parameters (i.e. the number of planned spacer occurrences). We also note that in this framework “true negatives” (“all correctly not reported matches that do not actually align”) is not a sensical or useful definition. By extension we can not compute certain common performance metrics such as specificity.

We acknowledge that a limitation of this system is the lack of a complete set of all positives for datasets too large to be exhaustively searched (using sassy and indelfree.sh bruteforce mode). In such cases, the positive set is essentially the union of all valid tool reported alignments.

**Non-planned Match Rate Estimation:** Using the synthetic datasets allows us to estimate the frequency of these reported alignments as a dependency of distance metric and threshold, and of the search space size. As we control the exact details in the simulated runs, we generate a ground truth table, where the location, number of mismatches, spacer and contig identifiers of each planned spacer occurrence is recorded. By combining this ground truth with the different tool results (particularly the exhaustive ones), we are able to identify (and subsequently verify) any reported alignment - and record the number of valid (within distance threshold) alignment not explicitly planned (occurring in regions other than those in the simulation plan). We expect these non-planned matches to represent chance similarities arising from sequence composition and length. To estimate the rate of such non-planned matches, under a given distance threshold, we divide the number of validated non-planned matches by different representations of the total search space size: A metric “engulfing” both spacer and contig set sizes in pp (sum of spacer lengths \* sum of contig lengths), or the product of the number of spacers and contigs (e.g. per n spacer and m contigs). Realistically, total spacer length is negligible compared to total contig length, however the product of the number of spacers and contigs assumes every contig and spacer effect the search space equally.  
We quantified non-planned match rates using the exhaustive search tools under varying hamming (indelfree bruteforce) and edit (sassy) distance thresholds (1 - 5). For large datasets (where the use of exhaustive search tools is too computationally prohibitive), we either reduced the threshold the range (1-3) if possible, although for the full set (fraction_1, the semi-synthetic set, and the largest of the simulated runs) we resorted to use the aggregation of the non exhaustive tools (namely Bowtie1 and blastn) results as proxy. Complete analysis and methodology details are provided in Supplementary Note 3.

### 3.6 Computational Resource and Runtime Tracking

While the primary focus of this study was to evaluate the ability of each tool to accurately identify spacer-protospacer matches, computational resource usage are a limiting factor for certain dataset sizes. This is particularly relevant for the exhaustive tools (Sassy and indelfree.sh in bruteforce mode), which have high computational costs.

The CLI benchmarking tool we developed utilises hyperfine<sup>[44](#ref-Peter_hyperfine_2023)</sup> for local execution of tools (suitable for smaller datasets), and a SLURM (Simple Linux Utility for Resource Management<sup>[45](#ref-SLURM_2002)</sup>) based method (suitable for larger datasets ran on high-performance-compute clusters), where we captured the resource usage via SLURM’s built-in accounting system (`sacct`), accessed through a custom Python wrapper. For consistency, all analyses herein were performed using the SLURM tracking approach. Both approaches allow us to capture detailed resource usage metrics, including wall clock time, CPU time, and peak memory usage, which are critical for understanding the practical feasibility of each tool under different conditions. All SLURM job logs were retained for reproducibility and are available in the Zenodo repository. All tools were allocated the same CPU and memory resources (64 threads, 512 GB RAM) to ensure a fair comparison, and the same maximum wall time limit (72 hours) was applied to all runs. If a tool exceeded the wall time limit, it was terminated and marked as “timed out” for that dataset, and no results were recorded for that run. The only exception to this was blastn for the fraction_1 set, which was allowed to run to completion as it is the main point of reference with regards to historical use of (any) tool for this task.  
We note that memory (RAM) tracking for some java tools (particularly BBMap suite tools such as indelfree.sh) can seem to use all available memory as the java virtual machine may not explicitly report cleared, unused memory, in a way visible to the SLURM accounting system. This means that the SLURM reported peak memory usage may not indicate the actual minimal requirement of a tool. For tools that require generating an index file (or any additional obligatory steps and commands) prior the actual search/scan/alignment command, we include the index construction time (or the additional commands) in the total runtime of a tool as these represents real computational cost, though for production use with repeated searches, index construction might constitute a one-time cost.

### 3.7 Versioning and Reproducibility

Versioning and Reproducibility {#sec-reproducibility}All tools were installed and managed using conda<sup>[46](#ref-conda)</sup> (via the mamba<sup>[47](#ref-mamba)</sup> package manager) in isolated environments. To prevent dependency conflicts and ensure reproducibility, most tools were installed in dedicated environments. Environment activation time was excluded from performance measurements to focus on actual tool runtime. The exact versions and configurations of all tools were recorded in environment files, allowing for exact replication of our testing environment.All benchmarks were performed on a standardized high-performance computing node to ensure consistency. The hardware environment consisted of a dual-socket system equipped with two AMD EPYC 7543 32-Core Processors, providing a total of 64 physical cores and 64 threads. The CPUs operated at a frequency of 3705.616 MHz with a 32 MB L3 cache. The system was equipped with 512 GB of RAM ($5.5 \times 10^{11}$ bytes) and utilized a SAMSUNG MZ1LB1T9HALS-00007 storage unit managed via an NFS file system.The software stack was deployed on Linux (kernel version 4.18.0-553.58.1.el8_10.x86_64) using an x86_64 architecture. Benchmarking scripts were executed using Python 3.10.19.

### 3.8 Extensibility

The benchmarking framework is designed to be expandable through the integration of new tools. Each tool/software configuration is saved as a separate JSON file, which includes the exact commands and conda/mamba environment it uses. This configuration files can use placeholder variables which the main benchmarking software replaces (according to the users’ CLI arguments) during execution (such as `{threads}`, `{contigs_file}`, `{spacers_file}`, `{output_dir}`, and `{results_dir}`). A new JSON file can be added manually or via `bench.utils.tool_commands:add_tool` function in a semi automated method.

## 4 Results

### 4.1 Selection of distance metric and threshold values

To investigate the effects of metric and threshold choice, we utilized both simulated and semi-synthetic datasets to quantify non-planned match rates. As noted in the methods (see Methods §sec-performance-calc), these non-planned matches are not considered “false positives” in the traditional sense, as they represent valid alignments within the distance threshold (verified independently of which tool reported them) that occur in regions not explicitly included in the simulation plan. In this section, we use these non-planned matches to estimate the expected “background noise” under different distance metrics (edit versus hamming), thresholds (at exact n distance or cumulative up to n distances), and search space sizes (based on simulation/dataset parameters).

To empirically assess the expected frequency of non-planned matches (valid alignments occurring in regions without planned spacer occurrences, see Methods §sec-performance-calc) under different distance metrics and thresholds, we analyzed both fully synthetic and semi-synthetic datasets. In this context the non-planned (validated) matches are considered as genuine sequence similarities arising from chance. We quantified non-planned match frequency using three metrics: 1. **Absolute count**: total number of verified non-planned alignments. 2. **Per spacer per Gbp target**: non-planned count divided by the number of spacers and the total contig length in Gbp ($\frac{\text{count}}{n_{\text{spacers}} \times \text{contig\_bp} / 10^9}$). This metric normalizes for dataset scale differences. 3. **Per Gbp**$^2$ search space: non-planned count divided by the product of total spacer length and total contig length in Gbp$^2$ ($\frac{\text{count}}{\text{spacer\_bp} \times \text{contig\_bp} / 10^{18}}$). This metric accounts for the full combinatorial search space.

Briefly, computational constraints limited exhaustive edit-distance search to smaller fully synthetic datasets. Using exhaustive tools (Sassy and indelfree), we observed substantially higher non-planned match rates when allowing indels (Figure <a href="#fig-background-normalization" class="quarto-xref">Figure 1</a>). At edit distance ≤3, non-planned match frequencies exceeded hamming distance ≤3 rates by approximately 5-10 fold depending on dataset. This increase reflects both biological permissiveness and the larger combinatorial space of edit distance relative to substitutions-only hamming distance. The search-space-normalized rates were relatively consistent across simulations, while very small thresholds (0 and ≤1) remained noisier due to low event counts.

<figure id="fig-background-normalization">

<figcaption>Figure 1: Non-planned alignment (“background”) frequency: mean ± SD across 4 fully synthetic simulations (50,000–100,000 spacers × 5,000–20,000 contigs). (A) Absolute count of non-planned matches at each distance threshold (0–5). (B) Non-planned matches normalized per spacer per Gbp of target sequence. (C) Non-planned matches normalized per Gbp<span class="math inline">^2</span> of total search space (spacer-bp × contig-bp). Blue bars: hamming distance (substitutions only). Orange bars: edit distance (substitutions + indels). Error bars: standard deviation across simulations. At hamming ≤3, the per Gbp<span class="math inline">^2</span> rate is approximately 40,000, while at edit ≤3 the rate is approximately 270,000–290,000 (a 6–8 fold increase). Full per-simulation breakdowns are provided in Supplementary Note 3.</figcaption>
</figure>

Similarly, we use the semi-synthetic dataset, combining the 3,826,979 real spacers (from iPHoP) with 421,431 fully-simulated contigs, as analogous to the “fraction_1” and use it to provide an estimate of the total amount of non-planned match. We note that this dataset size was too prohibitive for the exhaustive and edit-distance based tools, hence we rely on the aggregate of the non-exhaustive tools as a proxy for the total number of non-planned matches, which is likely an underestimate. Similarly, these results only estimate the hamming-based non-planned match rates. Specifically, the total amount of non-planned matches was 1 for 0 hamming threshold (i.e. exact match), 49 for an hamming distance of ≤1, 2,354 matches at ≤2 hamming distance, and 56,273 at ≤3 distance (see <a href="#tbl-semisynthetic-hamming3" class="quarto-xref">Table 2</a> for the results at hamming ≤3 compared to the fully synthetic simulation mean.)

<table style="width:98%;">
<colgroup>
<col style="width: 26%" />
<col style="width: 26%" />
<col style="width: 24%" />
<col style="width: 21%" />
</colgroup>
<thead>
<tr>
<th style="text-align: center;"></th>
<th style="text-align: center;">Fully-Simulated (mean ± SD, n=4)</th>
<th style="text-align: center;">Semi-synthetic (real spacers)</th>
<th style="text-align: center;">Ratio</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: center;"><strong>Non-planned matches (total)</strong></td>
<td style="text-align: center;">91.3 ± 64.5</td>
<td style="text-align: center;">56,273</td>
<td style="text-align: center;">n/a<br />
(different search space )</td>
</tr>
<tr>
<td style="text-align: center;"><strong>Per spacer per Gbp</strong></td>
<td style="text-align: center;">0.0013 ± 0.0001</td>
<td style="text-align: center;">0.0034</td>
<td style="text-align: center;">2.6×</td>
</tr>
<tr>
<td style="text-align: center;"><strong>Per Gbp</strong><span class="math inline">^2</span> <strong>search space</strong></td>
<td style="text-align: center;">40,314 ± 2,791</td>
<td style="text-align: center;">101,270</td>
<td style="text-align: center;">2.5×</td>
</tr>
</tbody>
</table>

Table 2: Comparison of non-planned match rates at hamming distance **≤3** between fully synthetic simulations and the semi-synthetic dataset. Absolute counts are not directly comparable due to different search space sizes. Normalized metrics show the semi-synthetic dataset has approximately 2.5× higher non-planned match rates, likely reflecting biological sequence composition features in real spacers not fully captured by simulated sequences (see Supplementary Note 3 for detailed methodology, per-threshold breakdowns, and caveats).

The approximately 2.5-fold higher normalized rates in the semi-synthetic dataset relative to fully synthetic simulations may stem from the real spacer sequences having compositional features (conserved motifs, non-uniform k-mer distributions, locus-specific GC variation) that increase chance similarity. Therefore, these values should be interpreted as qualitative indicators of relative (between edit vs hamming and at increasing thresholds) background alignment rather than exact predictive rates, as the simulated contigs only capture basic sequence features (namely GC% and length ranges), but not all structural features of real viral genomes.

We selected hamming distance ≤3 as the primary threshold for most subsequent analyses, but we acknowledge that for certain specific research questions and scenarios, an edit distance may be more appropriate, and we include some key analyses herein.

### 4.2 Tool performance across distance thresholds

<figure id="fig-tool-performance">

<figcaption>Figure 2: Tool recall (detection fraction of unique valid aligned regions) across distance thresholds. Upper row shows IMG/VR4 (fractions) results and lower row shows synthetic datasets; columns compare hamming and edit distance analyses. An asterix “*” is noted after tools that did failed or timed out, and the total number of sucessful runs (subsampled fraction for the IMG/VR sets, and independent measurements for the simulated set) are noted in brackets after each tool’name, and the value in each cell is the mean of those run specific recall values. Note - the values are at exact distance (unlike at a “max” distance), i.e. regions that aligned with distance n are not considered for distance n+1.</figcaption>
</figure>

Tool performance varied systematically with distance threshold (<a href="#fig-tool-performance" class="quarto-xref">Figure 2</a>). At distance 0 (exact matches), multiple tools achieved recall above 0.95 on both simulated and real datasets, including Bowtie1, Bowtie2, BLASTn, and MUMmer4. However, no single non-exhaustive tool identified all occurrences at this threshold. As mismatch tolerance increased, differences between tools became more pronounced.

Across both simulated and real datasets, Bowtie1 remained the highest-recall heuristic method at hamming distance ≤3, while tools optimized for different mapping contexts (notably StrobeAlign and MMseqs2) showed larger recall losses at higher mismatch levels. In edit-distance analyses, exhaustive Sassy provided the expected upper bound where runs were computationally feasible, while heuristic edit-capable tools showed variable sensitivity depending on dataset scale and mismatch tolerance.

Performance patterns were consistent between simulated and real datasets, though the smaller simulated dataset size and availability of exhaustive tools on more synthetic data enabled more precise recall measurement. Complete per-tool, per-threshold recall values are provided in Supplementary Table S5. In this regard, it is important to note that within the allocated or available total CPU time per job, the largest “real-dataset” img/vr fracion amenable to the exhustaive tools was 1% (sassy, edit) and 0.1% (indelfree_bruteforce, hamming). See section below regarding computioanl resource usage for more details.

Pairwise tool comparisons (“measruing the set-difference of reported alignments between each two tools) were consistent with these recall trends (Supplementary Figure S2), indicating the same alignemnts were reported by multiple tools at low distance values, albeit tool-specific differences quickly grew with the increase in distance.

### 4.3 Computational Resource Requirements and Scalability

Computational resource requirements vary dramatically across tools, reflecting fundamental differences in algorithmic approaches and trade-offs between sensitivity and efficiency (<a href="#fig-resource-usage" class="quarto-xref">Figure 3</a>). Understanding these resource requirements is essential for practical tool selection, particularly as CRISPR spacer and viral databases continue growing rapidly.

<figure id="fig-resource-usage">

<figcaption>Figure 3: Total CPU time scaling with dataset size (log-log scale). (A) Real IMG/VR4 subsampled datasets (fractions 0.001–1.0, approximately 10 Mbp to 18.9 Gbp target sequence). (B) Simulated datasets with varying numbers of spacers and contigs (see Supplementary Table S4 for dataset details). Marker shape and color encode tool identity. BLASTn runtime for fraction_1 was extrapolated from partial completion (1.69M of 3.83M spacers processed within the 72h wall time limit; see Supplementary Table S8 note).</figcaption>
</figure>

CPU-time trends in Figure <a href="#fig-resource-usage" class="quarto-xref">Figure 3</a> show large differences in practical scalability, with exhaustive tools typically requiring substantially longer runs even on smaller datasets. Missing points indicate tool-dataset combinations that did not produce a completed value for plotting (for example timeout, failed runs, or runs excluded from the plotted subset); the corresponding SLURM-derived records are provided in Supplementary Table S8 (**?@tbl-supp-resource-usage**), and the logs (stdout and stderr) for all runs are available in the Zenodo repository.

Among heuristic methods, Bowtie1, Bowtie2, and StrobeAlign showed the most favorable wall-time behavior across increasing dataset size, while MMseqs2 required substantially more memory. BLASTn remained intermediate in runtime but exceeded the 72-hour wall-time limit on fraction_1 under sensitivity-oriented parameters, so that value was extrapolated from the observed partial run (see Supplementary Table S8). As expected, exhaustive tools were markedly slower; notably, indelfree_bruteforce remained expensive even at intermediate subsamples, reinforcing its role as a validation-oriented method rather than a default choice. Supriginly, Sassy, despite being an exhaustive edit-distance based tool, showed more favorable runtime than indelfree_bruteforce (a strictly hamming distance based tool), reflecting that algorithmic constraints can be greatly alleviated by dedicated implementation optimizations.

With regards to memory usage (Supplementary Figure S9 **?@fig-supp-resource-memory**), SLURM peak values differed by up to one to two orders of magnitude among tools. This metric should be interpreted together with runtime because some tools reduce memory pressure by partitioning reference data or increasing disk I/O, which can increase wall time. Reported peak memory for Java-based tools (x-mapper and BBMap-suite tools such as indelfree.sh) may also be affected by JVM allocation behavior relative to SLURM RSS accounting.

The paired real/simulated design also helps separate different scaling effects. In simulated runs, both spacer and contig counts vary (Supplementary Table S4), while in IMG/VR4 runs the spacer set is fixed and only contig-side subsampling changes. This contrast suggests that, for some tools, peak memory is influenced more strongly by index structure and long-contig properties than by total contig-set size alone. Complete resource values and run-level details are provided in Supplementary Table S8 and the resource-usage notebook.

### 4.4 Performance as a function of query (spacer) abundance in reference database

In the fraction_1 analysis, we observed occurrence-dependent sensitivity effects in most tools, where spacers with very high occurrence in the target set showed reduced recall (Supplementary Figure S5 **?@fig-supp-real-recall**). We corroborated this behavior with a dedicated high-insertion simulation (`ns_500_nc_5000_HIGH_INSERTION_RATE`), where each spacer was inserted 100-2500 times in simulated contigs (Supplementary Figure S4 **?@fig-supp-sim-recall**). The strongest degradations were tool-specific and most visible at the highest occurrence bins. Importantly, ultra-high occurrence spacers (\>1000 occurrences) represented a very small fraction of matched spacers (\<0.01%; Supplementary Figure S8 **?@fig-supp-occurrence-distribution**), so this effect is measurable but limited in overall contribution to aggregate recall.

## 5 Discussion

Our analysis across fully synthetic, semi-synthetic, and real IMG/VR4 datasets demonstrates that algorithmic design assumptions substantially influence spacer-protospacer detection sensitivity. The quantitative comparison of ten tools, including two exhaustive methods, enables evidence-based recommendations while revealing systematic differences rooted in the computational problems each tool was designed to solve.

### 5.1 Distance metric choice and practical thresholds

The empirical non-planned match rates measured in this study provide quantitative grounds for threshold selection. In the semi-synthetic dataset (3,826,979 real spacers searched against 421,431 simulated contigs), the number of validated non-planned matches rose sharply with distance: 1 at exact match, 49 at hamming distance \$\$1, 2,354 at \$\$2, and 56,273 at \$\$3 (<a href="#tbl-semisynthetic-hamming3" class="quarto-xref">Table 2</a>; Supplementary Table S7). Analyses of fully synthetic datasets using exhaustive tools further showed that edit distance \$\$3 produced approximately 6-8 fold more non-planned matches than hamming distance \$\$3 at the same numeric threshold (<a href="#fig-background-normalization" class="quarto-xref">Figure 1</a>; Supplementary Table S6). These quantitative observations, combined with the biological evidence reviewed in the introduction for substitution-dominant phage escape mutations<sup>[23](#ref-Deveau2008)–[26](#ref-Schelling2023)</sup>, support hamming distance \$\$3 as an appropriate default threshold for spacer-protospacer matching in the context of host-MGE interaction inference.

The practical implications of these rates are scale-dependent. For small datasets, such as a single or few phage genomes (total target sequence below approximately 1 Gbp) searched against spacers extracted from several host genomes, the per-search-space-normalized rates suggest that non-related matches at hamming \$\$3 are infrequent. For large-scale host assignment analyses where the total search space approaches or exceeds the scale tested here (approximately 129 Mbp spacer sequence $\times$ 4.3 Gbp contig sequence), the expected number of non-planned matches should inform downstream interpretation. At such scales, post-hoc filtering – for example, requiring multiple supporting spacers per host-virus pair, incorporating phylogenetic context, or applying stricter distance thresholds – may be warranted to mitigate the accumulation of chance matches. Edit distance metrics may be appropriate under narrowly defined conditions: when analyzing data from sequencing platforms with elevated indel error rates (e.g., Oxford Nanopore R9 chemistry), when explicitly characterizing escape mutation types in controlled experimental systems, or when predicting gene-editing off-targets where gapped alignments are biologically relevant. Outside of these scenarios, the substantially higher non-planned match rates associated with edit distance (6-8 fold at \$\$3) argue against its routine use for inferring host-MGE interactions from natural populations.

### 5.2 Tool performance and algorithmic considerations

A central observation is that tools differ in sensitivity not because of implementation deficiencies, but because their algorithms solve different computational problems. Edit/affine-based tools (Bowtie2, Minimap2, BLASTn, Sassy) and hamming-based tools (Bowtie1, indelfree.sh) will, by definition, identify different match sets when applied at the same numeric threshold, as they operate under different distance definitions. This fundamental distinction must be considered when interpreting and comparing results. Within the hamming distance analyses, Bowtie1 consistently achieved the highest recall among heuristic methods across both dataset types and distance thresholds (<a href="#fig-tool-performance" class="quarto-xref">Figure 2</a>). Performance patterns were notably consistent between simulated and real datasets (Supplementary Figure S10 **?@fig-supp-sim-consistency**), supporting the validity of our simulation framework and suggesting these findings generalize across database compositions.

At higher mismatch levels, inter-tool differences became more pronounced. Tools originally designed for mapping short reads against single-source reference assemblies (StrobeAlign, Minimap2) showed steeper recall declines, consistent with seed-selection heuristics that penalize high-frequency k-mers or that assume reference uniqueness. Similarly, we observed occurrence-dependent sensitivity effects (see <a href="#sec-abundance-performance" class="quarto-xref">Section 4.4</a>), where spacers with many valid targets in the reference showed reduced detection rates in most heuristic tools. While spacers with extremely high occurrence frequency (\$\>$1000 targets) represented a negligible fraction of the total matched set ($\<\$0.01%; Supplementary Figure S8 **?@fig-supp-occurrence-distribution**), the effect was measurable across a broader range of occurrence frequencies and may matter for analyses specifically targeting broadly conserved genomic regions.

The computational resource analysis reveals substantial differences in scalability (<a href="#fig-resource-usage" class="quarto-xref">Figure 3</a>; Supplementary Figure S9 **?@fig-supp-resource-memory**). While index construction is often a one-time upfront cost, we included it in the measured resource budget because it represents real computational expenditure. Among heuristic tools, Bowtie1 combined high recall with favorable runtime scaling, whereas BLASTn-short under sensitivity-oriented parameters exceeded the 72-hour wall-time limit on the full fraction_1 dataset. The practical feasibility of exhaustive tools depends strongly on dataset scale. For targeted analyses of specific host-phage systems or comparisons among defined lineages, where the total subject sequence is typically well below 1 Gbp and the number of query spacers ranges from tens to thousands, exhaustive tools such as Sassy and indelfree.sh bruteforce remain tractable and provide the advantage of guaranteed complete detection. In contrast, for metagenomic datasets where the combined contig set routinely exceeds 1 Gbp and spacer sets may contain millions of sequences, the computational cost of exhaustive search becomes prohibitive. For such analyses, heuristic tools – particularly Bowtie1 – offer near-perfect recall at hamming distance \$\$3 with orders-of-magnitude lower resource requirements, making them the practical choice. Sassy, notably, showed more favorable runtime than indelfree_bruteforce despite being an exhaustive edit-distance method, demonstrating that dedicated algorithmic optimizations can substantially improve the practical feasibility of exhaustive approaches. These resource constraints directly affect feasibility as CRISPR spacer databases continue growing – from 366,799 unique spacers in 2017<sup>[33](#ref-Shmakov_2017)</sup> to 3,835,942 in 2023<sup>[34](#ref-camargo_img_vr4_2023)</sup> – and heuristic tools with favorable scaling properties enable routine analysis at current and projected database sizes.

### 5.3 Biological interpretation of spacer-protospacer matches

Beyond tool performance, the biological interpretation of identified matches requires careful consideration of several confounding scenarios. Low-complexity regions, whether in the viral target set or in the spacer set itself (where non-CRISPR repeated sequences may occasionally be misclassified as spacers), can produce spurious matches independently of tool choice. While certain tools employ internal filtering heuristics, a separate complexity masking step using tools such as DUST<sup>[8](#ref-Morgulis_2006)</sup> or similar approaches (e.g. ldust, BBDuk) prior to searching remains advisable.

Horizontal gene transfer (HGT) can spread conserved sequences across unrelated MGEs, creating genuine sequence similarities that do not reflect direct host-MGE interactions. Kosmopoulos et al. described a case where a transposon-mediated transfer of a phage lysin gene to the host genome produced a verifiable sequence match<sup>[48](#ref-Kosmopoulos_2023)</sup>, illustrating that an unknown fraction of observed alignments may reflect HGT rather than CRISPR-mediated interactions. Self-targeting events, where spacers match host genomic sequences, have been estimated to have putative exogenous origin (e.g. prophages) in approximately 50%<sup>[9](#ref-Stern_2010)</sup> to 80%<sup>[33](#ref-Shmakov_2017)</sup> of cases, though the rate of true non-defense self-targeting appears to be low<sup>[33](#ref-Shmakov_2017)</sup>. Additional complexities include CRISPR systems encoded on non-chromosomal replicons such as plasmids<sup>[49](#ref-Maier_2018),[50](#ref-Zhang_2025)</sup>, inter-species spacer acquisition in archaea<sup>[51](#ref-Turgeman_Grott_2018)</sup>, and phage-encoded mini-arrays that can interfere with host CRISPR immunity<sup>[52](#ref-Shmakov_2023)</sup>.

The source and context of spacer data also merits consideration. Spacers extracted from assembled CRISPR arrays carry additional information – notably their position within the array and the observation that co-located spacers originate from the same host. Mitrofanov et al.<sup>[53](#ref-Mitrofanov2025)</sup> identified system subtype-dependent patterns of spacer loss, and Vink et al.<sup>[54](#ref-Vink2021)</sup> reported that most matched spacers in their analysis contained three or fewer mismatched nucleotides, consistent with our general recommendation of hamming distance \$\$3. Vink et al. also observed strand-targeting preferences varying by CRISPR subtype (e.g. Type I-E and Type II systems preferentially target template strands), information that could enhance post-search verification when combined with sequence orientation and coding potential analysis.

### 5.4 Study limitations

Several aspects of this study merit explicit acknowledgment. The synthetic datasets, while configured to match GC content, length distributions, and complexity characteristics of real data (Supplementary Figure S6 **?@fig-supp-spacer-comparison**), do not fully capture all structural features of biological sequences, including locus-specific composition biases, coding constraints, and evolutionary signatures. The approximately 2.5-fold higher normalized non-planned match rates observed in the semi-synthetic dataset relative to the fully synthetic simulations (<a href="#tbl-semisynthetic-hamming3" class="quarto-xref">Table 2</a>) likely reflect such compositional features in real spacer sequences, and the non-planned match rates reported here should accordingly be interpreted as order-of-magnitude guides rather than precise predictive values.

Exhaustive tools (Sassy, indelfree.sh bruteforce) could not be run on the full HQ benchmark dataset due to computational constraints. Our comparisons on the largest datasets therefore rely on the aggregate of heuristic tool results as a proxy for the complete positive set. For the same reason, edit-distance-based exhaustive analysis was limited to the smaller fully synthetic datasets. Tools were configured to maximize sensitivity rather than exhaustively exploring all parameter combinations, and further optimization for specific use cases may be possible. We also note that our analyses focused on assembled contigs from Illumina-derived data; for other sequencing technologies or raw-read-based workflows, additional considerations regarding sequencing error profiles may apply.

### 5.5 Conclusion

This work demonstrates that tool choice in spacer-protospacer matching carries measurable consequences for downstream biological inference. Based on the empirical evidence presented, we suggest Bowtie1 at hamming distance \$\$3 as a practical default for large-scale analyses of host-MGE interactions through CRISPR spacers, given its combination of high sensitivity, computational efficiency, and consistency across dataset scales. For analyses requiring tolerance beyond 3 substitutions, indelfree.sh in indexed mode extends the hamming distance range without incurring the elevated non-planned match rates associated with edit distance. Exhaustive tools such as Sassy remain appropriate for small-scale analyses – including experimental studies of individual host-phage systems, lineage-specific comparisons, or validation of heuristic tool outputs – where the total subject sequence is modest (below approximately 1 Gbp) and computational resources permit complete enumeration. Edit distance metrics may be warranted under specific conditions: when analyzing low-accuracy long-read data where sequencing-induced indels cannot be excluded, when investigating escape mutation mechanisms in controlled experimental systems where the mutation type itself is informative, or when working in gene-editing off-target prediction contexts where gapped alignments have established biological relevance. For all other cases, hamming distance \$\$3 captures the dominant mode of protospacer divergence (substitutions) while maintaining a favorable ratio of genuine to non-planned matches. We emphasize that regardless of tool choice, post-hoc verification of alignment quality, complexity filtering, and contextual biological analysis remain important components of any spacer-protospacer matching workflow.

## 6 Code and data availability

All code generated for this study can be found in the git repository: [github.com/UriNeri/spacer_matching_bench](https://github.com/UriNeri/spacer_matching_bench). All data generated in this project (tool results and the analysed datasets, any sequence data, SLURM logs etc) are available on Zenodo<sup>[39](#ref-zenodo_doi)</sup>.

## 7 Acknowledgements

Work conducted by the U.S. DOE Joint Genome Institute (https://ror.org/04xm1d337) (SR, UN, APC and BB), a DOE Office of Science User Facility, is supported by the Office of Science of the U.S. DOE operated under Contract DE-AC02-05CH11231.

We would like to thank the following people for their helpful feedback and suggestions: Uri Gophna, Georg Rath, and Ragnar Groot Koerkamp for valuable discussions on distance metrics and tool performance.

<span class="csl-left-margin">1. </span><span class="csl-right-inline">Mojica, F. J. M., Díez-Villaseñor, C., García-Martínez, J. & Soria, E. [Intervening sequences of regularly spaced prokaryotic repeats derive from foreign genetic elements](https://doi.org/10.1007/s00239-004-0046-3). *Journal of Molecular Evolution* **60**, 174–182 (2005).</span>

<span class="csl-left-margin">2. </span><span class="csl-right-inline">Koonin, K. S., Eugene V. AND Makarova. [Evolutionary plasticity and functional versatility of CRISPR systems](https://doi.org/10.1371/journal.pbio.3001481). *PLOS Biology* **20**, 1–19 (2022).</span>

<span class="csl-left-margin">3. </span><span class="csl-right-inline">Makarova, K. S. *et al.* Evolutionary classification of CRISPR–cas systems: A burst of class 2 and derived variants. *Nature Reviews Microbiology* **18**, 67–83 (2020).</span>

<span class="csl-left-margin">4. </span><span class="csl-right-inline">Edwards, R. A., McNair, K., Faust, K., Raes, J. & Dutilh, B. E. [Computational approaches to predict bacteriophage–host relationships](https://doi.org/10.1093/femsre/fuv048). *FEMS Microbiology Reviews* **40**, 258–272 (2015).</span>

<span class="csl-left-margin">5. </span><span class="csl-right-inline">Jiang, F. & Doudna, J. A. [CRISPR–Cas9 structures and mechanisms](https://doi.org/10.1146/annurev-biophys-062215-010822). *Annual Review of Biophysics* **46**, 505–529 (2017).</span>

<span class="csl-left-margin">6. </span><span class="csl-right-inline">Soto-Perez, P. *et al.* [CRISPR-Cas System of a Prevalent Human Gut Bacterium Reveals Hyper-targeting against Phages in a Human Virome Catalog](https://doi.org/10.1016/j.chom.2019.08.008). *Cell Host & Microbe* **26**, 325–335.e5 (2019).</span>

<span class="csl-left-margin">7. </span><span class="csl-right-inline">Frith, M. C. [A new repeat-masking method enables specific detection of homologous sequences](https://doi.org/10.1093/nar/gkq1212). *Nucleic Acids Research* **39**, e23–e23 (2010).</span>

<span class="csl-left-margin">8. </span><span class="csl-right-inline">Morgulis, A., Gertz, E. M., Schäffer, A. A. & Agarwala, R. [A fast and symmetric DUST implementation to mask low-complexity DNA sequences](https://doi.org/10.1089/cmb.2006.13.1028). *Journal of Computational Biology* **13**, 1028–1040 (2006).</span>

<span class="csl-left-margin">9. </span><span class="csl-right-inline">Stern, A., Keren, L., Wurtzel, O., Amitai, G. & Sorek, R. [Self-targeting by CRISPR: Gene regulation or autoimmunity?](https://doi.org/10.1016/j.tig.2010.05.008) *Trends in Genetics* **26**, 335–340 (2010).</span>

<span class="csl-left-margin">10. </span><span class="csl-right-inline">Biswas, A., Gagnon, J. N., Brouns, S. J. J., Fineran, P. C. & and, C. M. B. [CRISPRTarget](https://doi.org/10.4161/rna.24046). *RNA Biology* **10**, 817–827 (2013).</span>

<span class="csl-left-margin">11. </span><span class="csl-right-inline">Altschul, S. F., Gish, W., Miller, W., Myers, E. W. & Lipman, D. J. Basic local alignment search tool. *J. Mol. Biol.* **215**, 403–410 (1990).</span>

<span class="csl-left-margin">12. </span><span class="csl-right-inline">Shah, N., Nute, M. G., Warnow, T. & Pop, M. [Misunderstood parameter of NCBI BLAST impacts the correctness of bioinformatics workflows](https://doi.org/10.1093/bioinformatics/bty833). *Bioinformatics* **35**, 1613–1614 (2018).</span>

<span class="csl-left-margin">13. </span><span class="csl-right-inline">Madden, T. L., Busby, B. & Ye, J. [Reply to the paper: Misunderstood parameters of NCBI BLAST impacts the correctness of bioinformatics workflows](https://doi.org/10.1093/bioinformatics/bty1026). *Bioinformatics* **35**, 2699–2700 (2018).</span>

<span class="csl-left-margin">14. </span><span class="csl-right-inline">Needleman, S. B. & Wunsch, C. D. [A general method applicable to the search for similarities in the amino acid sequence of two proteins](https://doi.org/10.1016/0022-2836(70)90057-4). *Journal of Molecular Biology* **48**, 443–453 (1970).</span>

<span class="csl-left-margin">15. </span><span class="csl-right-inline">Smith, T. F. & Waterman, M. S. [Identification of common molecular subsequences](https://doi.org/10.1016/0022-2836(81)90087-5). *Journal of Molecular Biology* **147**, 195–197 (1981).</span>

<span class="csl-left-margin">16. </span><span class="csl-right-inline">Myers, G. [A fast bit-vector algorithm for approximate string matching based on dynamic programming](https://doi.org/10.1145/316542.316550). *Journal of the ACM* **46**, 395–415 (1999).</span>

<span class="csl-left-margin">17. </span><span class="csl-right-inline">Langmead, B., Trapnell, C., Pop, M. & Salzberg, S. L. [Ultrafast and memory-efficient alignment of short DNA sequences to the human genome](https://doi.org/10.1186/gb-2009-10-3-r25). *Genome Biology* **10**, R25 (2009).</span>

<span class="csl-left-margin">18. </span><span class="csl-right-inline">Langmead, B. & Salzberg, S. L. [Fast gapped-read alignment with Bowtie 2](https://doi.org/10.1038/nmeth.1923). *Nature Methods* **9**, 357–359 (2012).</span>

<span class="csl-left-margin">19. </span><span class="csl-right-inline">Li, H. [Minimap2: Pairwise alignment for nucleotide sequences](https://doi.org/10.1093/bioinformatics/bty191). *Bioinformatics* **34**, 3094–3100 (2018).</span>

<span class="csl-left-margin">20. </span><span class="csl-right-inline">Sahlin, K. [Strobealign: Flexible seed size enables ultra-fast and accurate read alignment](https://doi.org/10.1186/s13059-022-02831-7). *Genome Biology* **23**, 260 (2022).</span>

<span class="csl-left-margin">21. </span><span class="csl-right-inline">Steinegger, M. & Söding, J. [MMseqs2 enables sensitive protein sequence searching for the analysis of massive data sets](https://doi.org/10.1038/nbt.3988). *Nature Biotechnology* **35**, 1026–1028 (2017).</span>

<span class="csl-left-margin">22. </span><span class="csl-right-inline">Jain, M., Olsen, H. E., Paten, B. & Akeson, M. [The oxford nanopore MinION: Delivery of nanopore sequencing to the genomics community](https://doi.org/10.1186/s13059-016-1103-0). *Genome biology* **17**, 239 (2016).</span>

<span class="csl-left-margin">23. </span><span class="csl-right-inline">Deveau, H. *et al.* [Phage response to CRISPR-encoded resistance in *streptococcus thermophilus*](https://doi.org/10.1128/JB.01412-07). *Journal of Bacteriology* **190**, 1390–1400 (2008).</span>

<span class="csl-left-margin">24. </span><span class="csl-right-inline">Semenova, E. *et al.* [Interference by clustered regularly interspaced short palindromic repeat (CRISPR) RNA is governed by a seed sequence](https://doi.org/10.1073/pnas.1104144108). *Proceedings of the National Academy of Sciences* **108**, 10098–10103 (2011).</span>

<span class="csl-left-margin">25. </span><span class="csl-right-inline">Fineran, P. C. *et al.* [Degenerate target sites mediate rapid primed CRISPR adaptation](https://doi.org/10.1073/pnas.1400071111). *Proceedings of the National Academy of Sciences* **111**, E1629–E1638 (2014).</span>

<span class="csl-left-margin">26. </span><span class="csl-right-inline">Schelling, M. A., Nguyen, G. T. & Sashital, D. G. [CRISPR-Cas effector specificity and cleavage site determine phage escape outcomes](https://doi.org/10.1371/journal.pbio.3002065). *PLoS Biology* **21**, e3002065 (2023).</span>

<span class="csl-left-margin">27. </span><span class="csl-right-inline">Lee, H., Popodi, E., Tang, H. & Foster, P. L. [Rate and molecular spectrum of spontaneous mutations in the bacterium escherichia coli as determined by whole-genome sequencing](https://doi.org/10.1073/pnas.1210309109). *Proceedings of the National Academy of Sciences of the United States of America* **109**, E2774–83 (2012).</span>

<span class="csl-left-margin">28. </span><span class="csl-right-inline">Kucukyildirim, S., Miller, S. F. & Lynch, M. [Low base‐substitution mutation rate and predominance of insertion‐deletion events in the acidophilic bacterium acidobacterium capsulatum](https://doi.org/10.1002/ece3.8429). *Ecology and Evolution* **11**, 17609–17614 (2021).</span>

<span class="csl-left-margin">29. </span><span class="csl-right-inline">Hatfull, G. F. & Hendrix, R. W. [Bacteriophages and their genomes](https://doi.org/10.1016/j.coviro.2011.06.009). *Current opinion in virology* **1**, 298–303 (2011).</span>

<span class="csl-left-margin">30. </span><span class="csl-right-inline">Ha, A. D. & Denver, D. R. [Comparative genomic analysis of 130 bacteriophages infecting bacteria in the genus pseudomonas](https://doi.org/10.3389/fmicb.2018.01456). *Frontiers in Microbiology* **9**, (2018).</span>

<span class="csl-left-margin">31. </span><span class="csl-right-inline">Paez-Espino, D. *et al.* [CRISPR immunity drives rapid phage genome evolution in *streptococcus thermophilus*](https://doi.org/10.1128/mBio.00262-15). *mBio* **6**, e00262–15 (2015).</span>

<span class="csl-left-margin">32. </span><span class="csl-right-inline">Kupczok, A. *et al.* [Rates of mutation and recombination in siphoviridae phage genome evolution over three decades](https://doi.org/10.1093/molbev/msy027). *Molecular Biology and Evolution* **35**, 1147–1159 (2018).</span>

<span class="csl-left-margin">33. </span><span class="csl-right-inline">Shmakov, S. A. *et al.* [The CRISPR spacer space is dominated by sequences from species-specific mobilomes](https://doi.org/10.1128/mbio.01397-17). *mBio* **8**, (2017).</span>

<span class="csl-left-margin">34. </span><span class="csl-right-inline">Camargo, A. P. *et al.* [IMG/VR v4: An expanded database of uncultivated virus genomes within a framework of extensive functional, taxonomic, and ecological metadata](https://doi.org/10.1093/nar/gkac1037). *Nucleic Acids Research* **51**, D733–D743 (2023).</span>

<span class="csl-left-margin">35. </span><span class="csl-right-inline">Dion, M. B. *et al.* [Streamlining CRISPR spacer-based bacterial host predictions to decipher the viral dark matter](https://doi.org/10.1093/nar/gkab133). *Nucleic Acids Research* **49**, 3127–3138 (2021).</span>

<span class="csl-left-margin">36. </span><span class="csl-right-inline">Zhang, R. *et al.* [SpacePHARER: Sensitive identification of phages from CRISPR spacers in prokaryotic hosts](https://doi.org/10.1093/bioinformatics/btab222). *Bioinformatics* **37**, 3364–3366 (2021).</span>

<span class="csl-left-margin">37. </span><span class="csl-right-inline">Roux, A. P. A. C., Simon AND Camargo. [iPHoP: An integrated machine learning framework to maximize host prediction for metagenome-derived viruses of archaea and bacteria](https://doi.org/10.1371/journal.pbio.3002083). *PLOS Biology* **21**, 1–26 (2023).</span>

<span class="csl-left-margin">38. </span><span class="csl-right-inline">Camargo, A. P. *et al.* [Identification of mobile genetic elements with <span class="nocase">geNomad</span>](https://doi.org/10.1038/s41587-023-01953-y). *Nature Biotechnology* **42**, 1303–1312 (2024).</span>

<span class="csl-left-margin">39. </span><span class="csl-right-inline">Neri, U., Pedro Camargo, A., Roux, S. & Brian, B. Supplementary data for CRISPR spacer-protospacer matching benchmarks \[data set\]. doi:[10.5281/zenodo.15171878](https://doi.org/10.5281/zenodo.15171878).</span>

<span class="csl-left-margin">40. </span><span class="csl-right-inline">Edgar, R. C. [PILER-CR: Fast and accurate identification of CRISPR repeats](https://doi.org/10.1186/1471-2105-8-18). *BMC Bioinformatics* **8**, 18 (2007).</span>

<span class="csl-left-margin">41. </span><span class="csl-right-inline">Bland, C. *et al.* [CRISPR Recognition Tool (CRT): A tool for automatic detection of clustered regularly interspaced palindromic repeats](https://doi.org/10.1186/1471-2105-8-209). *BMC Bioinformatics* **8**, 209 (2007).</span>

<span class="csl-left-margin">42. </span><span class="csl-right-inline">Daily, J. [Parasail: SIMD c library for global, semi-global, and local pairwise sequence alignments](https://doi.org/10.1186/s12859-016-0930-z). *BMC Bioinformatics* **17**, 81 (2016).</span>

<span class="csl-left-margin">43. </span><span class="csl-right-inline">Šošić, M. & Šikić, M. [Edlib: A c/c ++ library for fast, exact sequence alignment using edit distance](https://doi.org/10.1093/bioinformatics/btw753). *Bioinformatics* **33**, 1394–1395 (2017).</span>

<span class="csl-left-margin">44. </span><span class="csl-right-inline">Peter, D. [<span class="nocase">hyperfine</span>](https://github.com/sharkdp/hyperfine). (2023).</span>

<span class="csl-left-margin">45. </span><span class="csl-right-inline">Jette, M., Dunlap, C., Garlick, J. & Grondona, M. [SLURM: Simple linux utility for resource management](https://www.osti.gov/biblio/15002962). (2002).</span>

<span class="csl-left-margin">46. </span><span class="csl-right-inline">conda contributors. [<span class="nocase">conda: A system-level, binary package and environment manager running on all major operating systems and platforms.</span>](https://github.com/conda/conda)</span>

<span class="csl-left-margin">47. </span><span class="csl-right-inline">mamba-org. [Mamba: The Fast Cross-Platform Package Manager](https://github.com/mamba-org/mamba).</span>

<span class="csl-left-margin">48. </span><span class="csl-right-inline">Kosmopoulos, J. C., Campbell, D. E., Whitaker, R. J. & Wilbanks, E. G. [Horizontal gene transfer and CRISPR targeting drive phage-bacterial host interactions and coevolution in “pink berry” marine microbial aggregates](https://doi.org/10.1128/aem.00177-23). *Applied and Environmental Microbiology* **89**, (2023).</span>

<span class="csl-left-margin">49. </span><span class="csl-right-inline">Maier, L.-K. *et al.* [The nuts and bolts of the haloferax CRISPR-cas system i-b](https://doi.org/10.1080/15476286.2018.1460994). *RNA Biology* **16**, 469–480 (2018).</span>

<span class="csl-left-margin">50. </span><span class="csl-right-inline">Zhang, A.-N. *et al.* [CRISPR-cas spacer acquisition is a rare event in human gut microbiome](https://doi.org/10.1016/j.xgen.2024.100725). *Cell Genomics* **5**, 100725 (2025).</span>

<span class="csl-left-margin">51. </span><span class="csl-right-inline">Turgeman-Grott, I. *et al.* [Pervasive acquisition of CRISPR memory driven by inter-species mating of archaea can limit gene transfer and influence speciation](https://doi.org/10.1038/s41564-018-0302-8). *Nature Microbiology* **4**, 177–186 (2018).</span>

<span class="csl-left-margin">52. </span><span class="csl-right-inline">Shmakov, S. A. *et al.* [Widespread CRISPR-derived RNA regulatory elements in CRISPR-cas systems](https://doi.org/10.1093/nar/gkad495). *Nucleic Acids Research* **51**, 8150–8168 (2023).</span>

<span class="csl-left-margin">53. </span><span class="csl-right-inline">Mitrofanov, A., Beisel, C. L., Baumdicker, F., Alkhnbashi, O. S. & Backofen, R. Comprehensive analysis of CRISPR array repeat mutations reveals subtype-specific patterns and links to spacer dynamics. *bioRxiv* (2025) doi:[10.1101/2025.04.02.646798](https://doi.org/10.1101/2025.04.02.646798).</span>

<span class="csl-left-margin">54. </span><span class="csl-right-inline">Vink, J. N. A., Baijens, J.-H. L. & Brouns, S. J. J. PAM-repeat associations and spacer selection preferences in single and co-occurring CRISPR-cas systems. *Genome Biology* (2021) doi:[10.1186/s13059-021-02495-9](https://doi.org/10.1186/s13059-021-02495-9).</span>